In [1]:
import time
import pandas as pd
import polars as pl

csv_url = "https://raw.githubusercontent.com/gakudo-ai/open-datasets/refs/heads/main/customers-100000.csv"

# Cargamos dataset en Polars
df_polars = pl.read_csv(csv_url)

# Cargamos dataset también en Pandas para comparar
df_pandas = pd.read_csv(csv_url)

print(f"Dimensiones en Polars: {df_polars.shape}")
print(f"Dimensiones en Pandas: {df_pandas.shape}")

Dimensiones en Polars: (100000, 12)
Dimensiones en Pandas: (100000, 12)


In [2]:
# SELECCIÓN DE DATOS POR FILAS Y COLUMNAS
print("--- Filtrado en Pandas ---")
# Pandas: Sintaxis de corchetes anidados: menos interpretabilidad
res_pandas = df_pandas[df_pandas["Country"] == "Chile"][["First Name", "Last Name", "Company"]]
print(res_pandas.head(3))

print("\n--- Filtrado en Polars ---")
# Polars: Métodos encadenados y explícitos: mayor interpretabilidad
res_polars = (
    df_polars
    .filter(pl.col("Country") == "Chile")
    .select(["First Name", "Last Name", "Company"])
)
print(res_polars.head(3))

--- Filtrado en Pandas ---
    First Name Last Name                   Company
153    Kristen   Rosales  Booth, Goodman and Ramos
537      Brent   Vincent            Ellison-Conway
703     Darren      Hess                  Hale Inc

--- Filtrado en Polars ---
shape: (3, 3)
┌────────────┬───────────┬──────────────────────────┐
│ First Name ┆ Last Name ┆ Company                  │
│ ---        ┆ ---       ┆ ---                      │
│ str        ┆ str       ┆ str                      │
╞════════════╪═══════════╪══════════════════════════╡
│ Kristen    ┆ Rosales   ┆ Booth, Goodman and Ramos │
│ Brent      ┆ Vincent   ┆ Ellison-Conway           │
│ Darren     ┆ Hess      ┆ Hale Inc                 │
└────────────┴───────────┴──────────────────────────┘


In [3]:
# COMPARATIVA DE TIEMPOS DE EJECUCIÓN EN OPERACIONES INTENSIVAS: AGRUPACIÓN Y AGREGACIÓN
# 1. Medimos el tiempo en Pandas
start_time = time.time()
agg_pandas = df_pandas.groupby("Country")["Index"].count()
pandas_time = time.time() - start_time

# 2. Medimos el tiempo en Polars
start_time = time.time()
agg_polars = (
    df_polars
    .group_by("Country")
    .agg(pl.len().alias("Total_Customers")) # pl.len() es la forma moderna de contar en Polars
)
polars_time = time.time() - start_time

print(f"Tiempo de ejecución en Pandas: {pandas_time:.5f} segundos")
print(f"Tiempo de ejecución en Polars: {polars_time:.5f} segundos")
print(f"¡Polars es {pandas_time / polars_time:.2f} veces más rápido en esta operación!")

Tiempo de ejecución en Pandas: 0.01429 segundos
Tiempo de ejecución en Polars: 0.04704 segundos
¡Polars es 0.30 veces más rápido en esta operación!


In [4]:
# PRUEBA DE ESTRÉS: QUIÉN GANA CUANDO REALMENTE MANEJAMOS MUCHOS DATOS
print("--- PRUEBA DE ESTRÉS: 5 MILLONES DE FILAS ---")
print("Multiplicando los datos 50 veces...\n")

# Multiplicamos los DataFrames
df_pandas_large = pd.concat([df_pandas] * 50, ignore_index=True)
df_polars_large = pl.concat([df_polars] * 50)

print(f"Nuevas dimensiones: {df_polars_large.shape[0]} filas")

# 1. Medimos Pandas
start_time = time.time()
agg_pandas = df_pandas_large.groupby("Country")["Index"].count()
pandas_time = time.time() - start_time

# 2. Medimos Polars
start_time = time.time()
agg_polars = (
    df_polars_large
    .group_by("Country")
    .agg(pl.len().alias("Total_Customers"))
)
polars_time = time.time() - start_time

print(f"\nTiempo Pandas (5M filas): {pandas_time:.5f} segundos")
print(f"Tiempo Polars (5M filas): {polars_time:.5f} segundos")

if polars_time < pandas_time:
    print(f"¡Ahora sí! Polars es {pandas_time / polars_time:.2f} veces más rápido.")
else:
    print("Pandas sigue ganando (podría pasar...).")

--- PRUEBA DE ESTRÉS: 5 MILLONES DE FILAS ---
Multiplicando los datos 50 veces...

Nuevas dimensiones: 5000000 filas

Tiempo Pandas (5M filas): 0.73790 segundos
Tiempo Polars (5M filas): 0.33807 segundos
¡Ahora sí! Polars es 2.18 veces más rápido.


In [5]:
# EVALUACIÓN PEREZOSA CON collect()
# Usamos pl.scan_csv en lugar de pl.read_csv para "activar" este modo de trabajo en Polars
query_perezosa = (
    pl.scan_csv(csv_url)
    .filter(pl.col("Country") == "Japan")
    .group_by("City")
    .agg(pl.len().alias("Clientes_en_Japon"))
    .sort("Clientes_en_Japon", descending=True)
)

print("¿Qué tipo de objeto es query_perezosa?")
print(type(query_perezosa))
# En un objeto LazyFrame, ¡los datos aún no se han procesado! solo se ha anotado el proceso a realizar

# Ahora sí, le pedimos a Polars que ejecute de forma óptima todo el plan trazado
resultado_final = query_perezosa.collect()

print("\nResultado Final:")
print(resultado_final.head(5))

¿Qué tipo de objeto es query_perezosa?
<class 'polars.lazyframe.frame.LazyFrame'>

Resultado Final:
shape: (5, 2)
┌──────────────────┬───────────────────┐
│ City             ┆ Clientes_en_Japon │
│ ---              ┆ ---               │
│ str              ┆ u32               │
╞══════════════════╪═══════════════════╡
│ Curtisfurt       ┆ 2                 │
│ East Perry       ┆ 2                 │
│ New Annettemouth ┆ 1                 │
│ Port Jamesmouth  ┆ 1                 │
│ Port Francis     ┆ 1                 │
└──────────────────┴───────────────────┘
